Using POS-Tagging, Dependency-Parsing on Samsung reviews to find the qualitative aspects of top 5 / 10 features

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import spacy

In [204]:
path = r'C:\Users\arnig\Documents\Coding_2024\python_work\UpGrad\DS_C70\Specialization_NLP\Syntactic-Processing\Syntactic-Processing_upgrad\POS Tagging Case Study\Dataset\Samsung.txt'
with open(path, mode='r', encoding='utf-8') as f_open:
    reviews_data = f_open.read()

print(type(reviews_data))
print(len(reviews_data))

<class 'str'>
7488235


In [205]:
# Splitting into sentences
reviews = reviews_data.split('\n')
reviews[:10]

["I feel so LUCKY to have found this used (phone to us & not used hard at all), phone on line from someone who upgraded and sold this one. My Son liked his old one that finally fell apart after 2.5+ years and didn't want an upgrade!! Thank you Seller, we really appreciate it & your honesty re: said used phone.I recommend this seller very highly & would but from them again!!",
 'nice phone, nice up grade from my pantach revue. Very clean set up and easy set up. never had an android phone but they are fantastic to say the least. perfect size for surfing and social media. great phone samsung',
 'Very pleased',
 'It works good but it goes slow sometimes but its a very good phone I love it',
 'Great phone to replace my lost phone. The only thing is the volume up button does not work, but I can still go into settings to adjust. Other than that, it does the job until I am eligible to upgrade my phone again.Thaanks!',
 'I originally was using the Samsung S2 Galaxy for Sprint and wanted to retu

In [206]:
from spacy import displacy
nlp = spacy.load('en_core_web_sm')

In [208]:
# Top features
nlp_pos = spacy.load('en_core_web_sm', disable=['parser', 'ner'])
nouns = []
for doc in tqdm(reviews):
    tokens = nlp_pos(doc)
    nouns.extend(token.lemma_ for token in tokens if token.pos_ == 'NOUN')

nouns = pd.Series(nouns).value_counts(normalize= True)

100%|██████████| 46355/46355 [02:12<00:00, 348.79it/s]


In [209]:
# Top 10 features from reviews
top = 10
features = nouns.head(top).index.values
features

array(['phone', 'battery', 'product', 'time', 'screen', 'card', 'price',
       'problem', 'camera', 'app'], dtype=object)

In [210]:
# Filtering reviews based on presence of top feature words
feature_reviews = {feature: [doc for doc in reviews if feature in doc] for feature in features}

Filtering qualities based on pos and dependency of qualifiers with feature words.

In [240]:
def check_adverb(token, feature):
    an = [str(a) for a in list(token.ancestors)]
    if token.pos_ == 'ADV' and feature in an:
        return token.lemma_
    else:
        return ''
    
def check_adjective(token, feature):
    an = [str(a) for a in list(token.ancestors)]
    if token.pos_ == 'ADJ' and feature in an:
        return token.lemma_
    else:
        return ''
    
def check_verb_dependency(token, feature):
    qualities = []
    if token.pos_ in ['VERB', 'AUX'] and feature in [ch.lemma_ for ch in token.children]:
        quality = [f'{token.lemma_} {a.lemma_}' for a in token.children if a.dep_ in ['amod', 'acomp', 'advmod', 'advcl', 'neg', 'det']]
        if len(quality):
            qualities.extend(quality)
    if len(qualities):
        return qualities
    else:
        return ['']

In [241]:
nlp_dep = spacy.load('en_core_web_sm', disable=['ner'])
feature_qualities = {}

for feature, docs in feature_reviews.items():
    qualities = []
    for doc in tqdm(docs):
        for tok in nlp_dep(doc):

            adv = check_adverb(tok, feature)                    # returns words

            adj = check_adjective(tok, feature)                 # returns words

            verb_dep = check_verb_dependency(tok, feature)      # return lists

            quality = [adv] + [adj] + verb_dep

            if len(quality):
                qualities.extend([q for q in quality if q])

    feature_qualities[feature] = pd.Series(qualities)

100%|██████████| 5061/5061 [00:54<00:00, 92.80it/s] 


Concat features and qualifiers into a dataframe

In [262]:
FeatureQuality = pd.concat([pd.Series(feature_qualities[key].value_counts(normalize= True).head(100), name=key) 
                            for key in feature_qualities.keys()], axis=1)

In [264]:
FeatureQuality[FeatureQuality.isna()] = 0
FeatureQuality

,phone,battery,product,time,screen,card,price,problem,camera,app
great,0.098530,0.009367,0.166477,0.010875,0.033807,0.000000,0.196406,0.000000,0.068952,0.007021
good,0.066265,0.012489,0.181300,0.018488,0.030332,0.003462,0.157039,0.003826,0.064113,0.005015
new,0.029082,0.038359,0.008837,0.011963,0.002528,0.041048,0.005135,0.007652,0.000000,0.003009
very,0.027546,0.007583,0.062714,0.020663,0.024329,0.000000,0.039795,0.004304,0.025000,0.000000
nice,0.027177,0.005352,0.016534,0.002175,0.027488,0.000000,0.012837,0.000000,0.020565,0.000000
...,...,...,...,...,...,...,...,...,...,...
open tube,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.003009
transfer easily,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.003009
run smoothly,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.003009
keep personally,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.003009


Feature importance by vector length

In [265]:
feature_imp = {}
for col in FeatureQuality.columns:
    feature_imp[col] = np.linalg.norm(FeatureQuality[col].values)

for key in sorted(feature_imp, key= lambda x: feature_imp[x], reverse= True):
    print(f'{key} = {feature_imp[key]}')

product = 0.3108429499245852
price = 0.2653970991557072
problem = 0.18431052395902434
time = 0.14986172833020792
card = 0.13872557834433466
phone = 0.13751374049509257
camera = 0.1366495314038157
screen = 0.11761111229913324
battery = 0.09244025668414482
app = 0.06178079676804211


Frobenius norm of Feature Quality matrix

In [266]:
F = FeatureQuality.values
print('Frobenius norm: ', np.linalg.norm(F))

Frobenius norm:  0.5536336492703617


Cosine similarity between features in the context of feature qualifiers

In [272]:
def cosine_sim(x, y):
    return np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y))

In [273]:
start_pos = 0
similarities = np.zeros((F.shape[1], F.shape[1]))
for i in range(F.shape[1]):
    for j in range(F.shape[1]):
        if i != j:
            s = cosine_sim(F[:, i], F[:, j])
            similarities[i][j] = s
        else:
            similarities[i][j] = 0
    start_pos += 1

In [274]:
similarities

array([[0.        , 0.3002145 , 0.81504093, 0.23023003, 0.50028315,
        0.10615354, 0.89939722, 0.08135459, 0.75500829, 0.196327  ],
       [0.3002145 , 0.        , 0.18033204, 0.33291021, 0.27791114,
        0.18536304, 0.21164746, 0.14424118, 0.25183852, 0.13771253],
       [0.81504093, 0.18033204, 0.        , 0.1744777 , 0.41646856,
        0.02481111, 0.83247985, 0.03326614, 0.69214321, 0.12385737],
       [0.23023003, 0.33291021, 0.1744777 , 0.        , 0.15068109,
        0.04424008, 0.17974235, 0.12602066, 0.15732862, 0.0587319 ],
       [0.50028315, 0.27791114, 0.41646856, 0.15068109, 0.        ,
        0.04155563, 0.45158198, 0.10863073, 0.45624819, 0.10428188],
       [0.10615354, 0.18536304, 0.02481111, 0.04424008, 0.04155563,
        0.        , 0.02670463, 0.07380854, 0.02512099, 0.06499701],
       [0.89939722, 0.21164746, 0.83247985, 0.17974235, 0.45158198,
        0.02670463, 0.        , 0.02611766, 0.75645856, 0.15692265],
       [0.08135459, 0.14424118, 0.0332661

Maximum and minimum cosine similarities

In [ ]:
max_val = similarities[0][0]
min_val = similarities[0][1]        # because 1st value = 0

for i in range(similarities.shape[0]):
    for j in range(similarities.shape[1]):

        if max_val < similarities[i][j]:
            max_idx = [i, j]
            max_val = similarities[i][j]

        if similarities[i][j] != 0 and min_val > similarities[i][j]:
            min_idx = [i, j]
            min_val = similarities[i][j]

print(f'Max cosine similarity: {FeatureQuality.columns[max_idx[0]]}, {FeatureQuality.columns[max_idx[1]]} = {max_val:.3f}')
print(f'Min cosine similarity: {FeatureQuality.columns[min_idx[0]]}, {FeatureQuality.columns[min_idx[1]]} = {min_val:.3f}')

Max cosine similarity: phone, price = 0.899
Min cosine similarity: product, card = 0.025


Qualifier words for each feature type

In [270]:
for col in FeatureQuality.columns:
    print(col)
    print(FeatureQuality.loc[FeatureQuality[col] != 0, col].sort_values(ascending=False).index.values)
    print('-'*80)

phone
['great' 'good' 'new' 'very' 'nice' 'excellent' 'smart' 'unlocked' 'old'
 'be great' 'ever' 'awesome' 'first' 'be not' 'work great' 'amazing'
 'really' 'be good' 'be amazing' 'flip' 'well' 'best' 'just' 'basic'
 'fast' 'be awesome' 'other' 'be fast' 'little' 'previous'
 'love absolutely' 'like really' 'work well' 'perfect' 'android'
 'work the' 'same' 'be perfect' 'have now' 'be unlocked' 'be new' 'so'
 'beautiful' 'small' 'be be' 'simple' 'work perfectly' 'last' 'overall'
 'cheap' 'international' 'dual' 'have not' 'far' 'be easy' 'be overall'
 'big' 'recommend highly' 'easy' 'wonderful' 'fantastic' 'work this'
 'solid' 'love really' 'only' 'be still' 'more' 'be far' 'work fine'
 'second' 'high' 'right' '-' 'mobile' 'be well' 'much' 'pretty' 'used'
 'large' 'be really' 'be nice' 'decent' 'get just' 'expensive' 'look new'
 'have ever' 'use not' 'cool' 'reliable' 'rugged' 'be otherwise'
 'be compatible' 'get when' 'love much' 'super' 'work not' 'be worth'
 'most' 'work far' 'still'

Dependencies are not analysed if spacy parser is disabled